# Otium

Дипломный проект: постановка задачи на разработку сортировки фильмов и сериалов по рейтингу зрителей и кинокритиков в стриминговом сервисе Otium.

Тетрадь фиксирует результаты работы: требования заказчика, диаграмму последовательности, проверку существующих REST API и спецификацию доработок Web Server, Films Server и Series Server.


## Описание проекта

К компании RANA.Software обратился стриминговый сервис **Otium**. Разработчики уже добавили фильтрацию по эксклюзивному контенту сервиса. Следующая задача — новая функция в разделе «Библиотека»: пользователь должен уметь сортировать объединённый список фильмов и сериалов по рейтингу зрителей и/или рейтингу кинокритиков.

Цель проекта — понять требования заказчика и поставить задачу разработчикам так, чтобы новую функцию можно было реализовать в существующей архитектуре API. Проект не предполагает написание серверного кода: результатом работы являются диаграмма последовательности и спецификация доработок REST API.

Работа состоит из четырёх шагов.

1. Изучить интервью с представителем Otium и проверить, насколько требования заказчика поняты.
2. Построить диаграмму последовательности новой функции.
3. Протестировать существующие API в Swagger.
4. Поставить задачу на разработку новой операции API.

Итоговые артефакты:

- диаграмма последовательности взаимодействия пользователя, UI Otium, Web Server, Films Server и Series Server;
- задача на разработку с описанием изменений в Web Server API, Films Server API и Series Server API.


## Исследуемый набор данных

Исходными данными проекта были не таблицы для статистического анализа, а каталог контента Otium и коллекция существующих запросов REST API. Каталог доступен через три сервиса:

- **Web Server API** — объединяет фильмы и сериалы для клиента, операция `GET /content/list`;
- **Films Server API** — каталог фильмов, операции `GET /films/list` и `GET /films`;
- **Series Server API** — каталог сериалов, операции `GET /series/list` и `GET /series`.

В ответах API уже есть рейтинг зрителей `rating`. Рейтинга кинокритиков `criticsRating` в текущих ответах нет, хотя по требованиям заказчика сортировка должна работать и по нему.

Примеры единиц каталога, с которыми велось исследование:

- фильм «Хосэ Каньон»: комедия, США, 1995, `rating` 9.4, `criticsRating` 7.4;
- фильм «Ребус Атлантиды»: боевик, 2007, `rating` 7.3, `criticsRating` 6.1;
- фильм «Вестибюль»: драма, 2005, `rating` 8.9, `criticsRating` отсутствует (`null`);
- фильм «В руках тьмы»: боевик, 2010, `rating` 6.8, `criticsRating` отсутствует (`null`);
- сериал «Индийский океан и я»: детский, Индонезия, 2015, 4 серии, `rating` 9.5, `criticsRating` 8.4.

Типовая карточка контента содержит идентификатор, тип (`film` / `series`), название, описание, ссылки на изображение и превью, жанр, признак рекомендации, детали выпуска и рейтинги. Документация API открывалась в Swagger; тестовые запросы направлялись на mock-сервер Postman.

Ячейка ниже проверяет, что исходные артефакты репозитория на месте.


In [1]:
from pathlib import Path

ROOT = Path('..').resolve()
print('Корень репозитория:', ROOT)
print()

expected = [
    'README.md',
    'Шаблон_диаграммы_последовательности.drawio',
    '_docx_extract.txt',
]

print('Проверка исходных артефактов:')
missing = []
for name in expected:
    path = ROOT / name
    status = 'есть' if path.exists() else 'НЕТ'
    size = path.stat().st_size if path.exists() else 0
    print(f'  [{status}] {name}  ({size} байт)')
    if not path.exists():
        missing.append(name)

print()
if missing:
    raise FileNotFoundError('Не найдены файлы: ' + ', '.join(missing))
print('Все исходные материалы проекта найдены.')


Корень репозитория: C:\Users\Roman\Yandex.Disk\Учеба\Дипломный проект Тихонова версия №1\Practicum_projects_Otium

Проверка исходных артефактов:
  [есть] README.md  (14122 байт)
  [есть] Шаблон_диаграммы_последовательности.drawio  (24554 байт)
  [есть] _docx_extract.txt  (22096 байт)

Все исходные материалы проекта найдены.


## Цели и основные пункты исследования

Цель проекта — выяснить, чего не хватает в текущем ПО и API Otium, чтобы пользователь мог сортировать библиотеку по рейтингам, и зафиксировать для разработчиков конкретные изменения трёх сервисов.

Основные пункты работы:

1. Изучить интервью с представителем Otium и сформулировать требование: в «Библиотеке» пользователь выбирает параметр сортировки (`rating` или `criticsRating`) и направление (`asc` / `desc`), после чего получает отсортированный список фильмов и сериалов.
2. Построить диаграмму последовательности: запрос списка с фильтрами и сортировкой → `GET /content/list` → параллельные запросы `GET /films/list` и `GET /series/list` → объединение, фильтрация и сортировка на Web Server → отображение карточек с обоими рейтингами.
3. Протестировать Web Server API, Films Server API и Series Server API в Swagger: параметры запросов, тело ответа, наличие или отсутствие нужных полей.
4. Ответить на вопросы исследования:
   - какую задачу пользователя нужно решить;
   - как её можно решить;
   - чего не хватает в ПО и интерфейсе;
   - позволяет ли текущий Web Server API решить задачу и что в нём должно измениться;
   - удовлетворяют ли Films Server API и Series Server API запросам Web Server API и что в них должно измениться.
5. Поставить задачу на разработку: добавить `criticsRating` в ответы Films Server и Series Server; в `GET /content/list` добавить query-параметры `sortBy` и `direction` и правило размещения элементов с `null` в конце списка.


## Шаг 1. Требование заказчика

По интервью с представителем Otium сформулировано требование и use case.

**Требование.** Пользователь должен выбрать в интерфейсе параметр сортировки (`rating` или `criticsRating`) и направление (`asc` / `desc`), после чего получить отсортированный список контента.

**Use case.** В разделе «Библиотека» пользователь видит объединённый список фильмов и сериалов. Он задаёт фильтры и/или режим сортировки. Система возвращает список согласно указанным данным. При сортировке по рейтингу кинокритиков контент без оценки (`criticsRating = null`) отображается в конце списка, чтобы приоритет имели карточки с валидными оценками.


In [2]:
requirement = [
    ('sortBy', 'string', 'опционально', 'rating — рейтинг зрителей; criticsRating — рейтинг кинокритиков'),
    ('direction', 'string', 'опционально', 'asc — по возрастанию; desc — по убыванию'),
]

print('Параметры сортировки, которые нужны клиенту')
print(f'{"Параметр":<12} {"Тип":<8} {"Обязательность":<14} Описание')
print('-' * 100)
for name, typ, req, desc in requirement:
    print(f'{name:<12} {typ:<8} {req:<14} {desc}')

print()
print('Правило null:')
print('  элементы без значения поля sortBy всегда в конце списка,')
print('  независимо от direction; порядок таких элементов стабильный.')


Параметры сортировки, которые нужны клиенту
Параметр     Тип      Обязательность Описание
----------------------------------------------------------------------------------------------------
sortBy       string   опционально    rating — рейтинг зрителей; criticsRating — рейтинг кинокритиков
direction    string   опционально    asc — по возрастанию; desc — по убыванию

Правило null:
  элементы без значения поля sortBy всегда в конце списка,
  независимо от direction; порядок таких элементов стабильный.


## Шаг 2. Диаграмма последовательности

Диаграмма хранится в файле `Шаблон_диаграммы_последовательности.drawio`.

Сценарий новой функции:

1. Пользователь запрашивает список фильмов и сериалов Otium с фильтрами и сортировкой.
2. UI Otium вызывает `GET {{WebServer}}/content/list`.
3. Web Server параллельно запрашивает `GET {{FilmsServer}}/films/list` и `GET {{SeriesServer}}/series/list`.
4. Films Server и Series Server читают свои хранилища и возвращают списки **с рейтингом зрителей и рейтингом кинокритиков**.
5. Web Server объединяет данные, применяет фильтр по жанру и сортировку по выбранному рейтингу.
6. UI показывает карточки с изображением обоих рейтингов.

Ячейка ниже читает файл диаграммы и выводит участников и сообщения.


In [3]:
import html
import re
import xml.etree.ElementTree as ET

drawio = ROOT / 'Шаблон_диаграммы_последовательности.drawio'
tree_root = ET.parse(drawio).getroot()


def cell_text(raw):
    text = html.unescape(raw or '')
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace(' ', ' ').replace('&nbsp;', ' ')
    return re.sub(r'\s+', ' ', text).strip()


print('Страницы файла:')
for i, diagram in enumerate(tree_root.findall('diagram'), start=1):
    print(f'  {i}. {diagram.get("name")}')

print()
print('Участники диаграммы последовательности:')
seen = set()
for cell in tree_root.iter('mxCell'):
    style = cell.get('style') or ''
    value = cell_text(cell.get('value'))
    if 'umlLifeline' in style:
        if 'umlActor' in style:
            name = 'Пользователь'
        elif value:
            name = value
        else:
            continue
    elif 'shape=cylinder' in style and value:
        name = f'Хранилище: {value}'
    else:
        continue
    if name not in seen:
        seen.add(name)
        print(f'  - {name}')

print()
print('Сообщения на диаграмме:')
for cell in tree_root.iter('mxCell'):
    if cell.get('edge') != '1':
        continue
    value = cell_text(cell.get('value'))
    if value:
        print(f'  → {value}')


Страницы файла:
  1. Страница 1

Участники диаграммы последовательности:
  - Пользователь
  - Web Server
  - Series Server
  - UI: Otium
  - Films Server
  - Хранилище: Фильмы
  - Хранилище: Сериалы

Сообщения на диаграмме:
  → Запрос списка фильмов и сериалов Otium (с фильтрами и сортировкой)
  → обработать данные и применить сортировку по рейтингу кинокритиков
  → 1. Объединение данных (фильмы и сериалы) 2.фильтр по жанру
  → обработать данные и применить сортировкупо по пользовательскому рейтингу
  → Данные сериалов (включая пользовательский рейтинг и рейтинг кинокритиков)
  → GET {{WebServer}} / content/list
  → Карточки с изображением обоих рейтингов
  → GET {{FilmsServer}} /films /list
  → список фильмов и сериалов (с учетом фильтров и сортировки)
  → Отображение списка фильмов и сериалов Otium
  → SELECT*FROM films
  → GET {{SeriesServer}} /series /list
  → Список фильмов (с рейтингами)
  → Данные фильмов (включая пользовательский рейтинг и рейтинг кинокритиков)
  → Список сериа

## Шаг 3. Тестирование существующих API

Документация открывалась в Swagger. Проверялись Web Server API, Films Server API и Series Server API: методы, параметры запросов и тело ответа.

Ответы на вопросы исследования:

| Вопрос | Ответ |
| --- | --- |
| Какую задачу пользователя нужно решить? | В «Библиотеке» отсортировать объединённый список фильмов и сериалов по рейтингу зрителей и/или кинокритиков. |
| Как можно решить задачу? | Расширить `GET /content/list` параметрами `sortBy` и `direction`, а в ответы трёх API добавить `criticsRating`. Сортировку выполнять на Web Server после объединения списков. |
| Чего не хватает в ПО и интерфейсе? | В UI нет выбора параметра и направления сортировки, на карточке нет рейтинга кинокритиков. |
| Позволяет ли текущий Web Server API решить задачу? | Нет. В `GET /content/list` нет `sortBy` и `direction`, в ответе нет `criticsRating`. |
| Удовлетворяют ли Films Server API и Series Server API запросам Web Server? | Нет. Поле `rating` уже есть, поля `criticsRating` нет. Без него Web Server не сможет ни отсортировать список по критикам, ни отдать рейтинг клиенту. |

Ячейка ниже воспроизводит исследуемый каталог и показывает, каких полей не хватает в текущих ответах.


In [4]:
catalog = [
    {'id': 201, 'type': 'series', 'title': 'Индийский океан и я', 'genre': ['kids'], 'rating': 9.5, 'criticsRating': 8.4},
    {'id': 101, 'type': 'film', 'title': 'Хосэ Каньон', 'genre': ['comedy'], 'rating': 9.4, 'criticsRating': 7.4},
    {'id': 102, 'type': 'film', 'title': 'Вестибюль', 'genre': ['drama'], 'rating': 8.9, 'criticsRating': None},
    {'id': 104, 'type': 'film', 'title': 'Ребус Атлантиды', 'genre': ['action'], 'rating': 7.3, 'criticsRating': 6.1},
    {'id': 103, 'type': 'film', 'title': 'В руках тьмы', 'genre': ['action'], 'rating': 6.8, 'criticsRating': None},
]

current_fields = ['id', 'type', 'title', 'description', 'imageUrl', 'previewUrl', 'recordUrl', 'genre', 'recommended', 'details', 'rating']
needed_fields = current_fields + ['criticsRating']

print('Поля карточки контента')
print('  есть сейчас :', ', '.join(current_fields))
print('  нужно добавить:', 'criticsRating')
print('  после доработки:', ', '.join(needed_fields))
print()

print(f'{"id":<6} {"type":<8} {"title":<24} {"rating":>6} {"criticsRating":>14}')
print('-' * 62)
for item in catalog:
    critics = 'null' if item['criticsRating'] is None else f"{item['criticsRating']:.1f}"
    print(f"{item['id']:<6} {item['type']:<8} {item['title']:<24} {item['rating']:>6.1f} {critics:>14}")

print()
print('Пробелы текущих API:')
print('  Web Server GET /content/list — нет query-параметров sortBy и direction')
print('  Web Server GET /content/list — нет поля criticsRating в ответе')
print('  Films Server GET /films/list и GET /films — нет поля criticsRating')
print('  Series Server GET /series/list и GET /series — нет поля criticsRating')


Поля карточки контента
  есть сейчас : id, type, title, description, imageUrl, previewUrl, recordUrl, genre, recommended, details, rating
  нужно добавить: criticsRating
  после доработки: id, type, title, description, imageUrl, previewUrl, recordUrl, genre, recommended, details, rating, criticsRating

id     type     title                    rating  criticsRating
--------------------------------------------------------------
201    series   Индийский океан и я         9.5            8.4
101    film     Хосэ Каньон                 9.4            7.4
102    film     Вестибюль                   8.9           null
104    film     Ребус Атлантиды             7.3            6.1
103    film     В руках тьмы                6.8           null

Пробелы текущих API:
  Web Server GET /content/list — нет query-параметров sortBy и direction
  Web Server GET /content/list — нет поля criticsRating в ответе
  Films Server GET /films/list и GET /films — нет поля criticsRating
  Series Server GET /serie

## Шаг 4. Задача на разработку

Спецификация лежит в `_docx_extract.txt`. Нужно доработать три сервиса.

**Films Server API**

- `GET /films/list` — добавить опциональное поле `criticsRating` (`number` / `null`, диапазон 0.0–10.0).
- `GET /films` — то же поле в карточке фильма.

**Series Server API**

- `GET /series/list` — добавить `criticsRating`.
- `GET /series` — добавить `criticsRating`.

**Web Server API**

- `GET /content/list` — добавить query-параметры `sortBy` (`rating` | `criticsRating`) и `direction` (`asc` | `desc`).
- В каждый элемент ответа пробросить `criticsRating`.
- Элементы с `null` по полю сортировки всегда в конце списка.
- Недопустимые значения параметров — ошибка `400 bad input parameter`.


In [5]:
spec_path = ROOT / '_docx_extract.txt'
spec = spec_path.read_text(encoding='utf-8')

operations = [
    ('1.1', 'Films Server', 'GET /films/list', 'добавить criticsRating'),
    ('1.2', 'Films Server', 'GET /films', 'добавить criticsRating'),
    ('2.1', 'Series Server', 'GET /series/list', 'добавить criticsRating'),
    ('2.2', 'Series Server', 'GET /series', 'добавить criticsRating'),
    ('3.1', 'Web Server', 'GET /content/list', 'добавить sortBy, direction и правило null'),
    ('3.2', 'Web Server', 'GET /content/list', 'добавить criticsRating в ответ'),
]

print('Операции из постановки задачи')
print(f'{"№":<6} {"Сервис":<15} {"Операция":<20} Изменение')
print('-' * 90)
for num, service, op, change in operations:
    print(f'{num:<6} {service:<15} {op:<20} {change}')

print()
print('Проверка текста спецификации:')
checks = [
    ('criticsRating', spec.count('criticsRating')),
    ('sortBy', spec.count('sortBy')),
    ('direction', spec.count('direction')),
    ('400', spec.count('400')),
]
for name, count in checks:
    print(f'  упоминаний {name}: {count}')

print()
print('Пример целевого запроса:')
print('  GET /content/list?sortBy=criticsRating&direction=desc')


Операции из постановки задачи
№      Сервис          Операция             Изменение
------------------------------------------------------------------------------------------
1.1    Films Server    GET /films/list      добавить criticsRating
1.2    Films Server    GET /films           добавить criticsRating
2.1    Series Server   GET /series/list     добавить criticsRating
2.2    Series Server   GET /series          добавить criticsRating
3.1    Web Server      GET /content/list    добавить sortBy, direction и правило null
3.2    Web Server      GET /content/list    добавить criticsRating в ответ

Проверка текста спецификации:
  упоминаний criticsRating: 38
  упоминаний sortBy: 16
  упоминаний direction: 9
  упоминаний 400: 1

Пример целевого запроса:
  GET /content/list?sortBy=criticsRating&direction=desc


Ниже воспроизведено правило сортировки, которое нужно реализовать на Web Server: сначала элементы с заполненным полем, затем элементы с `null` в исходном порядке.


In [6]:
def sort_content(items, sort_by, direction):
    reverse = direction == 'desc'
    with_value = [item for item in items if item.get(sort_by) is not None]
    without_value = [item for item in items if item.get(sort_by) is None]
    with_value.sort(key=lambda item: item[sort_by], reverse=reverse)
    return with_value + without_value


def print_list(title, items, field):
    print(title)
    print(f'  {"#":<4} {"title":<24} {field}')
    for i, item in enumerate(items, start=1):
        value = item.get(field)
        shown = 'null' if value is None else f'{value:.1f}'
        print(f'  {i:<4} {item["title"]:<24} {shown}')
    print()


print('Исходный объединённый список (как после слияния фильмов и сериалов):')
print_list('', catalog, 'criticsRating')

sorted_desc = sort_content(catalog, 'criticsRating', 'desc')
print_list('GET /content/list?sortBy=criticsRating&direction=desc', sorted_desc, 'criticsRating')

sorted_asc = sort_content(catalog, 'criticsRating', 'asc')
print_list('GET /content/list?sortBy=criticsRating&direction=asc', sorted_asc, 'criticsRating')

sorted_rating = sort_content(catalog, 'rating', 'desc')
print_list('GET /content/list?sortBy=rating&direction=desc', sorted_rating, 'rating')

nulls_last = all(item['criticsRating'] is not None for item in sorted_desc[:-2])
nulls_are_null = all(item['criticsRating'] is None for item in sorted_desc[-2:])
print('Проверка правила null для criticsRating desc:')
print('  заполненные оценки идут первыми:', nulls_last)
print('  элементы с null в конце:', nulls_are_null)
print('  порядок null стабильный: Вестибюль, затем В руках тьмы')


Исходный объединённый список (как после слияния фильмов и сериалов):

  #    title                    criticsRating
  1    Индийский океан и я      8.4
  2    Хосэ Каньон              7.4
  3    Вестибюль                null
  4    Ребус Атлантиды          6.1
  5    В руках тьмы             null

GET /content/list?sortBy=criticsRating&direction=desc
  #    title                    criticsRating
  1    Индийский океан и я      8.4
  2    Хосэ Каньон              7.4
  3    Ребус Атлантиды          6.1
  4    Вестибюль                null
  5    В руках тьмы             null

GET /content/list?sortBy=criticsRating&direction=asc
  #    title                    criticsRating
  1    Ребус Атлантиды          6.1
  2    Хосэ Каньон              7.4
  3    Индийский океан и я      8.4
  4    Вестибюль                null
  5    В руках тьмы             null

GET /content/list?sortBy=rating&direction=desc
  #    title                    rating
  1    Индийский океан и я      9.5
  2    Хосэ Ка

## Инструменты и библиотеки

Проект выполнен средствами системного анализа и тестирования API. Расчётных библиотек Python в исследовании не использовалось; ячейки тетради нужны для сверки артефактов, диаграммы, спецификации и правила сортировки.

- **Swagger UI** — изучение и тестирование Web Server API, Films Server API и Series Server API;
- **diagrams.net (draw.io)** — диаграмма последовательности (`Шаблон_диаграммы_последовательности.drawio`);
- **Postman Mock Server** — тестовые запросы к каталогу;
- **Microsoft Word** — постановка задачи на разработку;
- **Google Chrome** — работа со Swagger и диаграммами.


## Инструкции по развёртыванию и системные требования

Это не серверное приложение, а пакет аналитических артефактов. Чтобы открыть проект, достаточно файлов репозитория и документации API в браузере.

### Системные требования

- компьютер или ноутбук с Windows или macOS;
- доступ в интернет;
- браузер Google Chrome версии 70 и выше;
- для правки диаграммы — [diagrams.net](https://app.diagrams.net/) или приложение draw.io;
- для просмотра постановки задачи — Microsoft Word или текстовый файл `_docx_extract.txt`;
- для повторного тестирования API — Swagger UI и доступ к mock-серверу Otium.

### Как открыть проект

1. Клонировать или скачать репозиторий.
2. Открыть `Шаблон_диаграммы_последовательности.drawio` в diagrams.net.
3. Открыть `_docx_extract.txt` — постановка задачи разработчикам.
4. Открыть документацию трёх API в Swagger и повторить тестовые запросы из постановки задачи.

Установка сервера, базы данных и библиотек Python не требуется.


## Планы по доработке

Поставленная задача закрывает сортировку библиотеки по двум рейтингам. Дальше имеет смысл развивать продукт так:

- реализовать в интерфейсе «Библиотека» выбор параметра и направления сортировки и отображение обоих рейтингов на карточке;
- добавить пагинацию объединённого списка `GET /content/list`, чтобы сортировка работала на больших выборках;
- согласовать совместную работу уже существующей фильтрации (жанр, эксклюзивный контент) и новой сортировки в одном запросе;
- вынести правило «элементы с `null` всегда в конце» в общую библиотеку сортировки Web Server;
- покрыть параметры `sortBy` и `direction` автотестами, включая код ошибки 400;
- после реализации API провести приёмочное тестирование сценария из диаграммы на реальном, а не mock-сервере;
- при необходимости добавить сортировку по году выпуска и названию — по той же схеме, что и рейтинги.


## Вывод по проекту

Пользователю Otium в «Библиотеке» не хватает сортировки по рейтингу зрителей и кинокритиков. Текущий Web Server API такую задачу не решает: в `GET /content/list` нет параметров `sortBy` и `direction`, а в ответах нет поля `criticsRating`. Films Server API и Series Server API тоже не удовлетворяют будущим запросам Web Server: рейтинг зрителей уже есть, рейтинга кинокритиков нет.

По интервью сформулировано требование, сценарий зафиксирован на диаграмме последовательности, пробелы API проверены в Swagger, разработчикам поставлена конкретная задача. После доработки трёх сервисов клиент сможет запросить, например, `/content/list?sortBy=criticsRating&direction=desc` и получить объединённый список, в котором контент без оценки кинокритиков окажется в конце. Следующий шаг — реализация операции командой разработки и проверка сценария на рабочем API Otium.
